# W11 · From PyTorch to HIP — the AMD stack / 從 PyTorch 到 HIP

**English.** So far everything ran through PyTorch's ROCm backend. Real-time
texture decode needs custom kernels. HIP is AMD's CUDA-like C++ dialect; `hipcc`
compiles it for a specific GPU arch (`--offload-arch=gfx1201` for RDNA4,
`gfx1151` for RDNA3.5). We port the PEPS inference inner loop — grid sample +
MLP — into one **fused** kernel (`hip/fused_peps_kernel.hip`) and measure it.

**繁體中文.** 目前都走 PyTorch 的 ROCm 後端。即時材質解碼需要自訂 kernel。HIP 是
AMD 類 CUDA 的 C++ 方言;`hipcc` 針對特定 GPU 架構編譯。我們把 PEPS 推論內迴圈
(grid 取樣 + MLP)融合成單一 kernel 並量測。

In [1]:
import subprocess, shutil, os
os.chdir('..')  # repo root
have_hipcc = shutil.which('hipcc') is not None
arch = 'unknown'
if shutil.which('rocminfo'):
    out = subprocess.run(['rocminfo'], capture_output=True, text=True).stdout
    import re; m = re.search(r'gfx[0-9a-f]+', out)
    arch = m.group(0) if m else 'unknown'
print('hipcc:', have_hipcc, '| gfx arch:', arch)
print('RDNA4' if arch=='gfx1201' else 'RDNA3.5' if arch=='gfx1151' else '(other)')

hipcc: True | gfx arch: gfx1151
RDNA3.5


## 1. Build the fused kernel for this box / 為本機編譯融合 kernel

In [2]:
if have_hipcc:
    r = subprocess.run(['hipcc', f'--offload-arch={arch}',
                        'hip/fused_peps_kernel.hip', '-o', 'hip/fused_peps'],
                       capture_output=True, text=True)
    print('build ok' if r.returncode == 0 else r.stderr[:500])
else:
    print('hipcc not found on this box — see hip/README.md; run on Box A or B.')

build ok


## 2. Run and measure / 執行與量測

In [3]:
if have_hipcc and os.path.exists('hip/fused_peps'):
    r = subprocess.run(['hip/fused_peps', '262144', '200'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[:300])
else:
    print('(skipped — no hipcc / binary)')

fused sample+MLP: 2.3569 ms/iter  (262144 points, C=8 Hn=64)


## 3. Why fusion matters / 為何融合重要
The unfused path writes latents to global memory, then reads them back for the
MLP. Fusing keeps latents in registers — one kernel launch, no round-trip. On
real-time texture decode (millions of texels/frame) this is the difference
between hitting frame budget or not.

未融合路徑把 latent 寫到全域記憶體再讀回給 MLP。融合讓 latent 留在暫存器 —— 一次
kernel 啟動、無來回。對即時材質解碼(每幀數百萬 texel),這決定能否達到幀預算。